# GP Posterior Coverage Analysis

This notebook recomputes the GP cross-validation predictions and checks posterior interval coverage for held-out aggregated RSSI values. It intentionally writes a separate coverage CSV and does not modify the original CV MSE notebook.

In [ ]:
import os
import sys
import time
from pathlib import Path

os.environ['CUDA_VISIBLE_DEVICES'] = '1'

# os.environ.setdefault('JAX_PLATFORMS', 'cpu')

import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CWD = Path.cwd()
if CWD.name == 'milestone5':
    NOTEBOOK_DIR = CWD
elif (CWD / 'milestone5').exists() and CWD.name == 'code':
    NOTEBOOK_DIR = CWD / 'milestone5'
elif (CWD / 'code' / 'milestone5').exists():
    NOTEBOOK_DIR = CWD / 'code' / 'milestone5'
else:
    NOTEBOOK_DIR = CWD
CODE_DIR = NOTEBOOK_DIR.parent

sys.path.append(str(NOTEBOOK_DIR))
sys.path.append(str(CODE_DIR))

from gp import GaussianProcess, posterior_predictive_interval_summary
from gp_kernels import make_indoor_outdoor_mean, make_wifi_kernel


In [ ]:
CV_SPLIT_TYPES = ['rand', 'geo']
CV_N_SPLITS = 5
CV_CHAINS = 2
CV_SAMPLES = 211
CV_CALIBRATION_ITERS = 30
CV_BURN_IN = 10
CV_THIN = 2
CV_JITTER_FACTOR = 5
CV_KEY_SEED = 305
CV_PREDICT_METHOD = 'sequential'

COVERAGE_LEVELS = (0.5, 0.8, 0.9, 0.95)
COVERAGE_TARGET = 'observed'
COVERAGE_KEY_SEED = 1305
COVERAGE_RESULTS_PATH = NOTEBOOK_DIR / 'cv_gibbs_coverage_results.csv'
MSE_RESULTS_PATH = CODE_DIR / 'milestone4' / 'cv_gibbs_mse_results.csv'

In [ ]:

ap_form = "add"
ls_xy, ls_z, os_xyz, ls_t, os_t, ls_ap, os_ap = [0.05, 0.25, 20.0, 50.0, 0.0, 0.5, 0.0]
K = make_wifi_kernel(
    ap_form=ap_form,
    ls_xy=ls_xy,
    ls_z=ls_z,
    os_xyz=os_xyz,
    ls_t=ls_t,
    os_t=os_t,
    ls_ap=ls_ap,
    os_ap=os_ap,
)


def make_split_gp(X_split, y_split):
    return GaussianProcess(make_indoor_outdoor_mean(X_split, y_split), K)


In [ ]:
def evaluate_coverage_split(split_type, split_idx):
    split_dir = CODE_DIR / 'cv' / split_type

    X_cv_train = jnp.load(split_dir / f'X_train_{split_idx}.npy')
    y_cv_train = jnp.load(split_dir / f'y_train_{split_idx}.npy')
    obs_count_cv_train = jnp.load(split_dir / f'obs_count_train_{split_idx}.npy')
    obs_sse_cv_train = jnp.load(split_dir / f'obs_sse_train_{split_idx}.npy')

    X_cv_test = jnp.load(split_dir / f'X_test_{split_idx}.npy')
    y_cv_test = jnp.load(split_dir / f'y_test_{split_idx}.npy')
    obs_count_cv_test = jnp.load(split_dir / f'obs_count_test_{split_idx}.npy')

    split_gp = make_split_gp(X_cv_train, y_cv_train)
    split_gp.fit(X_cv_train, y_cv_train, obs_count_cv_train, obs_sse_cv_train)

    key_offset = 0 if split_type == 'rand' else CV_N_SPLITS

    start = time.time()
    chain = split_gp.gibbs(
        key=jr.PRNGKey(CV_KEY_SEED + key_offset + split_idx),
        chains=CV_CHAINS,
        samples=CV_SAMPLES,
        calibration_iters=CV_CALIBRATION_ITERS,
        jitter_factor=CV_JITTER_FACTOR,
    )
    fit_elapsed = time.time() - start

    cov_chains = chain[1][:, CV_BURN_IN::CV_THIN, :]
    start = time.time()
    pred_means, pred_vars = split_gp.predict(X_cv_test, cov_chains, method=CV_PREDICT_METHOD)
    predict_elapsed = time.time() - start

    y_hat = pred_means.mean(axis=0)
    mse = jnp.mean((y_cv_test - y_hat) ** 2)

    start = time.time()
    coverage = posterior_predictive_interval_summary(
        y_cv_test,
        pred_means,
        pred_vars,
        cov_chains=cov_chains,
        X_test=X_cv_test,
        obs_count_test=jnp.ones(y_cv_test.shape), #obs_count_cv_test,
        levels=COVERAGE_LEVELS,
        target=COVERAGE_TARGET,
        random_state=COVERAGE_KEY_SEED + key_offset + split_idx,
    )
    coverage_elapsed = time.time() - start

    return {
        'split_type': split_type,
        'split_idx': split_idx,
        'n_train': int(X_cv_train.shape[0]),
        'n_test': int(X_cv_test.shape[0]),
        'mse': float(mse),
        'fit_elapsed': fit_elapsed,
        'predict_elapsed': predict_elapsed,
        'coverage_elapsed' : coverage_elapsed,
        'jitter': float(split_gp.jitter),
        **coverage,
    }

Running the next cell refits all GP CV folds. It is expected to take about as long as the existing GP CV MSE workflow.

In [ ]:
coverage_results = []
for split_type in CV_SPLIT_TYPES:
    for split_idx in range(CV_N_SPLITS):
        print(f'Fitting {split_type} split {split_idx}')
        result = evaluate_coverage_split(split_type, split_idx)
        coverage_results.append(result)
        print(
            f"{split_type} split {split_idx}: "
            f"MSE={result['mse']:.4f}, "
            f"coverage_95={result['coverage_95']:.3f}, "
            f"width_95={result['avg_width_95']:.2f}, "
            f"fit={result['fit_elapsed']:.1f}s, "
            f"predict={result['predict_elapsed']:.1f}s",
            f"coverage={result['coverage_elapsed']:.1f}s"
        )

coverage_results_df = pd.DataFrame(coverage_results)
coverage_results_df.to_csv(COVERAGE_RESULTS_PATH, index=False)
coverage_results_df

In [ ]:
coverage_results_df[['split_type'] + [f'normal_coverage_{int(perc*100)}' for perc in COVERAGE_LEVELS]]

In [ ]:
coverage_results_df = pd.read_csv(COVERAGE_RESULTS_PATH)
coverage_cols = [f'coverage_{int(level * 100)}' for level in COVERAGE_LEVELS]
width_cols = [f'avg_width_{int(level * 100)}' for level in COVERAGE_LEVELS]

coverage_results_df.groupby('split_type')[coverage_cols + width_cols].mean()

In [ ]:
if MSE_RESULTS_PATH.exists():
    mse_results = pd.read_csv(MSE_RESULTS_PATH)
    coverage_with_mse = coverage_results_df.merge(
        mse_results[['split_type', 'split_idx', 'mse']],
        on=['split_type', 'split_idx'],
        how='left',
        suffixes=('', '_previous'),
    )
else:
    coverage_with_mse = coverage_results_df.copy()

coverage_with_mse

In [ ]:
def plot_coverage(split_type, level='95'):
    subset = coverage_results_df[coverage_results_df['split_type'] == split_type].sort_values('split_idx')
    nominal = int(level) / 100

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(subset['split_idx'], subset[f'coverage_{level}'], color=(157/255, 193/255, 131/255))
    ax.axhline(nominal, color='black', linestyle='--', linewidth=1.5, label=f'{nominal:.0%} nominal')
    ax.set_ylim(0, 1)
    ax.set_xlabel('Split index')
    ax.set_ylabel('Held-out coverage')
    ax.set_title(f'GP posterior predictive {level}% coverage ({split_type})')
    ax.legend()
    plt.show()


plot_coverage('rand', '95')
plot_coverage('geo', '95')

In [ ]:
def plot_coverage_vs_width(split_type, level='95'):
    subset = coverage_results_df[coverage_results_df['split_type'] == split_type].sort_values('split_idx')

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(subset[f'avg_width_{level}'], subset[f'coverage_{level}'], s=60, color=(107/255, 143/255, 81/255))
    for _, row in subset.iterrows():
        ax.annotate(int(row['split_idx']), (row[f'avg_width_{level}'], row[f'coverage_{level}']), xytext=(4, 4), textcoords='offset points')
    ax.axhline(int(level) / 100, color='black', linestyle='--', linewidth=1.5)
    ax.set_ylim(0, 1)
    ax.set_xlabel(f'Average {level}% interval width')
    ax.set_ylabel(f'{level}% coverage')
    ax.set_title(f'Coverage vs interval width ({split_type})')
    plt.show()


plot_coverage_vs_width('rand', '95')
plot_coverage_vs_width('geo', '95')